# Comprendre Claude — Exemples pratiques

Ce notebook illustre comment formuler des instructions précises en suivant le modèle **CONTEXTE / DONNÉES / OUTPUT / CONTRAINTES**.

Trois exemples concrets montrent la différence entre des demandes vagues et des spécifications claires, et comment cela facilite l'exécution correcte du code.

## Exemple 1 — Rapport mensuel énergie

### Instruction précise suivant le modèle CONTEXTE/DONNÉES/OUTPUT/CONTRAINTES :

```
CONTEXTE : Préparation d'un rapport mensuel pour le service énergie.
           Fichier source : ../data/consommation_energie.csv

DONNÉES  : Données couvrant 2023-2024. Quelques lignes présentent des valeurs négatives
           dans la colonne 'kwh' (erreurs capteurs) — à exclure avant traitement.

OUTPUT   : Script Python qui charge les données, applique les filtres,
           et génère ../output/rapport_mensuel.html contenant :
           - Un tableau récapitulatif par bâtiment
           - Un graphique en barres : consommation par bâtiment
           - Une courbe temporelle : kwh total par mois

CONTRAINTES : 
           - Ne pas modifier le CSV source
           - Utiliser Pandas + Plotly uniquement
           - Tous les commentaires en français
           - Style par défaut Plotly (pas de template personnalisé)
```

### Avantages d'une instruction précise :
- ✓ Étendue du traitement connue (pas de doutes sur les dates ou filtres)
- ✓ Technologie imposée (Pandas + Plotly → pas d'alternative hésitante)
- ✓ Format de sortie défini (HTML, fichier, structure)
- ✓ Contraintes claires → code plus robuste et maintenable

In [ ]:
import pandas as pd
import plotly.express as px
from pathlib import Path

# Chemins relatifs au notebook
data_dir = Path("../../data")
output_dir = Path("../../output")
output_dir.mkdir(exist_ok=True)

# Charger les données d'énergie
df_energie = pd.read_csv(data_dir / "consommation_energie.csv")

# Exclure les valeurs négatives (erreurs capteurs)
df_energie = df_energie[df_energie["kwh"] > 0].copy()

# Convertir la date en datetime pour les agrégations mensuelles
df_energie["date"] = pd.to_datetime(df_energie["date"])
df_energie["annee_mois"] = df_energie["date"].dt.to_period("M")

print(f"Données chargées : {len(df_energie)} lignes après filtrage")
print(f"Colonnes : {df_energie.columns.tolist()}")
print(f"Plage de dates : {df_energie['date'].min()} à {df_energie['date'].max()}")

In [ ]:
# Tableau récapitulatif : consommation totale par bâtiment
recap_batiment = df_energie.groupby("batiment")["kwh"].agg(["sum", "mean", "count"]).round(2)
recap_batiment.columns = ["Consommation totale (kWh)", "Moyenne (kWh)", "Nombre de mesures"]
print("\n--- Récapitulatif par bâtiment ---")
print(recap_batiment)

# Graphique 1 : consommation totale par bâtiment (bar chart)
data_batiment = df_energie.groupby("batiment")["kwh"].sum().reset_index()
fig1 = px.bar(
    data_batiment,
    x="batiment",
    y="kwh",
    title="Consommation énergétique par bâtiment",
    labels={"kwh": "Consommation (kWh)", "batiment": "Bâtiment"},
    color="kwh"
)
fig1.show()

In [ ]:
# Graphique 2 : évolution temporelle de la consommation mensuelle
data_mensuel = df_energie.groupby("annee_mois")["kwh"].sum().reset_index()
data_mensuel["annee_mois"] = data_mensuel["annee_mois"].astype(str)

fig2 = px.line(
    data_mensuel,
    x="annee_mois",
    y="kwh",
    title="Évolution mensuelle de la consommation totale",
    labels={"kwh": "Consommation (kWh)", "annee_mois": "Mois"},
    markers=True
)
fig2.update_xaxes(tickangle=-45)
fig2.show()

In [ ]:
# Générer le rapport HTML avec tableau + deux graphiques
html_content = f"""
<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8">
    <title>Rapport Énergétique Mensuel</title>
    <style>
        body {{ font-family: Arial, sans-serif; margin: 20px; background-color: #f5f5f5; }}
        h1 {{ color: #333; }}
        table {{ border-collapse: collapse; width: 100%; margin: 20px 0; background: white; }}
        th, td {{ border: 1px solid #ddd; padding: 12px; text-align: left; }}
        th {{ background-color: #4CAF50; color: white; }}
        tr:nth-child(even) {{ background-color: #f9f9f9; }}
    </style>
</head>
<body>
    <h1>Rapport Énergétique — {df_energie['date'].max().strftime('%B %Y')}</h1>
    
    <h2>Récapitulatif par bâtiment</h2>
    {recap_batiment.to_html()}
    
    <h2>Consommation par bâtiment</h2>
    {fig1.to_html(include_plotlyjs='cdn', div_id='fig1')}
    
    <h2>Évolution temporelle</h2>
    {fig2.to_html(include_plotlyjs='cdn', div_id='fig2')}
</body>
</html>
"""

output_file = output_dir / "rapport_mensuel.html"
output_file.write_text(html_content)
print(f"✓ Rapport généré : {output_file}")

## Exemple 2 — Analyse production

### Instruction précise :

```
CONTEXTE : Analyse de la qualité de production pour identifier les dérives
           par équipe et produit.
           Fichier source : ../data/production_industrielle.csv

DONNÉES  : Historique de production 2024 avec colonnes :
           quantite_produite, quantite_defauts, operateur (équipe).

OUTPUT   : Graphique scatter plot montrant la relation quantité vs défauts,
           avec une teinte par équipe et une trendline pour identifier
           les équipes hors norme.

CONTRAINTES :
           - Style par défaut Plotly
           - Afficher trendline linéaire sur chaque équipe
           - Pas de modification des CSV
```

### Points-clés :
- ✓ Problématique claire : identifier les dérives de qualité
- ✓ Type de visualisation imposé (scatter + trendline)
- ✓ Variable à visualiser (équipe)
- ✓ Sortie interprétable immédiatement

In [ ]:
# Charger les données de production
df_production = pd.read_csv(data_dir / "production_industrielle.csv")

# Calculer le taux de défaut (défauts / produits)
df_production["taux_defaut"] = (
    (df_production["quantite_defauts"] / df_production["quantite_produite"] * 100)
    .round(2)
)

print(f"Données production : {len(df_production)} enregistrements")
print(f"Équipes : {df_production['operateur'].unique().tolist()}")
print(f"\nTaux de défaut moyen par équipe :")
print(df_production.groupby("operateur")["taux_defaut"].mean().round(2))

In [ ]:
# Scatter plot : quantité vs défauts, coloré par équipe, avec trendline
fig_prod = px.scatter(
    df_production,
    x="quantite_produite",
    y="quantite_defauts",
    color="operateur",
    trendline="ols",  # Trendline linéaire par équipe
    title="Production vs Qualité — Analyse par équipe",
    labels={
        "quantite_produite": "Quantité produite",
        "quantite_defauts": "Nombre de défauts",
        "operateur": "Équipe"
    },
    height=500
)
fig_prod.show()

# Sauvegarder en HTML
fig_prod.write_html(output_dir / "analyse_production.html")
print("✓ Graphique sauvegardé : ../output/analyse_production.html")

## Exemple 3 — Détection d'anomalies capteurs

### Instruction précise :

```
CONTEXTE : Surveillance de capteurs de température en entrepôt.
           Objectif : détecter les dérives anormales (ex: T03 qui chaufferait).
           Fichier source : ../data/capteurs_temperature.csv

DONNÉES  : Série temporelle avec capteur_id (T01, T02, T03, T04) et temperature_c.
           Données de mars 2024, échantillonnage horaire.

OUTPUT   : Graphique ligne montrant la température de chaque capteur sur le temps.
           Légende claires, axes lisibles. Permet de repérer visuellement
           un capteur qui dérive.

CONTRAINTES :
           - Style par défaut Plotly
           - Une ligne par capteur (légende)
           - Axes en français
```

### Intérêt pédagogique :
- ✓ Problème de surveillance (détection d'anomalies visuellement)
- ✓ Série temporelle → visualisation ligne
- ✓ Comparaison multi-capteur → couleurs/légende essentielles

In [ ]:
# Charger les données capteurs
df_capteurs = pd.read_csv(data_dir / "capteurs_temperature.csv")

# Convertir timestamp en datetime
df_capteurs["timestamp"] = pd.to_datetime(df_capteurs["timestamp"])

print(f"Données capteurs : {len(df_capteurs)} enregistrements")
print(f"Capteurs : {sorted(df_capteurs['capteur_id'].unique())}")
print(f"Plage temporelle : {df_capteurs['timestamp'].min()} à {df_capteurs['timestamp'].max()}")
print(f"\nTempérature moyenne par capteur :")
print(df_capteurs.groupby("capteur_id")["temperature_c"].agg(["mean", "min", "max"]).round(2))

In [ ]:
# Graphique linéaire : température de chaque capteur en fonction du temps
fig_temp = px.line(
    df_capteurs,
    x="timestamp",
    y="temperature_c",
    color="capteur_id",
    title="Surveillance de température — Détection d'anomalies",
    labels={
        "timestamp": "Date et heure",
        "temperature_c": "Température (°C)",
        "capteur_id": "Capteur"
    },
    height=500
)
fig_temp.show()

# Sauvegarder
fig_temp.write_html(output_dir / "surveillance_capteurs.html")
print("✓ Graphique sauvegardé : ../output/surveillance_capteurs.html")

## Résumé — Vague vs Précis

| Aspect | Instruction vague | Instruction précise (CONTEXTE/DONNÉES/OUTPUT/CONTRAINTES) |
|--------|------|------|
| **Résultat** | Interprétations multiples, allers-retours | Code direct, une passe |
| **Technologie** | Incertitude (NumPy ou Pandas ? Matplotlib ou Plotly ?) | Imposée clairement |
| **Format de sortie** | À deviner | Spécifié (HTML, CSV, graphique…) |
| **Filtrage des données** | Ambiguïté sur les critères d'exclusion | Énumérés explicitement |
| **Maintenance** | Difficile : flou documentaire | Facile : spécifications tracées |
| **Robustesse** | Risques d'erreur | Contraintes appliquées d'emblée |

### À retenir :
1. **CONTEXTE** : pourquoi on fait cela ? (métier, objectif)
2. **DONNÉES** : sources, plage, anomalies connues
3. **OUTPUT** : format, destination, structure exacte
4. **CONTRAINTES** : technos, langues, conventions, limites

→ Une instruction précise **économise du temps** et **évite les ambiguïtés**.